# Serving an Open Model Yourself

From an empty machine to a traced, guarded chat endpoint. Nothing is
pre-installed and nothing is hidden behind a Makefile — every step below is a
command you run in order.

| | |
|---|---|
| **1** | Install the Hugging Face client, and list what Qwen actually publishes |
| **2** | Install vLLM and serve one of those models |
| **3** | Inference, and what it costs — GPU memory and latency, measured |
| **4** | Tracing with Phoenix, and how to read a trace |
| **5** | Guardrails with Guardrails AI — PII, NSFW, and what they miss |

### What you need

An NVIDIA GPU with **at least 40GB** of free VRAM for the default model, CUDA
drivers, and Python 3.10+.

You also need a **compiler and Python headers**. vLLM builds CUDA helpers on
first start, and without these it dies with `fatal error: Python.h: No such file
or directory` buried hundreds of lines into a log — measured on a clean Ubuntu
22.04 GPU box, where none of them were present:

```bash
sudo apt-get install -y python3-dev build-essential ninja-build
```

### Use a fresh environment

vLLM pins `torch`, `transformers` and `openai` hard. Installing it into an
environment you care about will move those versions under you.

```bash
python3 -m venv ~/serving-venv
source ~/serving-venv/bin/activate
pip install ipykernel
python -m ipykernel install --user --name serving --display-name "serving"
```

Then pick the **serving** kernel in Jupyter before running anything below.

---

## 1 · The Hugging Face client, and what Qwen publishes

`huggingface_hub` is the client for the model registry. It is small, pure
Python, and has nothing to do with the GPU — install it first because it is how
you find out what you are about to download.

In [ ]:
%pip install -q huggingface_hub ipywidgets


In [ ]:
import huggingface_hub
print("huggingface_hub", huggingface_hub.__version__)

### List the Qwen models

`HfApi.list_models` queries the registry live. No token is needed for public
models. Sorted by downloads, so the top of the list is what people actually run.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
# sort="downloads" already returns most-downloaded first. The `direction`
# argument existed in huggingface_hub 0.x and was REMOVED in 1.x -- passing it
# raises TypeError, which is the kind of breakage a pinned version prevents and
# a "pip install latest" notebook invites.
models = list(api.list_models(author="Qwen", sort="downloads", limit=40))

print(f"{len(models)} Qwen models, most downloaded first\n")
print(f"{'model':46} {'downloads':>12}  {'likes':>6}")
for m in models:
    print(f"{m.id:46} {m.downloads or 0:>12,}  {m.likes or 0:>6}")


### Narrow it to what you can actually serve

The list above mixes text models, embedding models, VL models and quantisations.
For a chat endpoint on one card, the axis that matters is **precision**, because
it decides whether the weights fit.

In [ ]:
import re

# Rough parameter count from the name: "27B" -> 27, "0.6B" -> 0.6
def params_b(model_id):
    m = re.search(r"(\d+(?:\.\d+)?)B", model_id)
    return float(m.group(1)) if m else None

def precision(model_id):
    low = model_id.lower()
    for tag in ("fp8", "awq", "gptq", "int4", "int8"):
        if tag in low:
            return tag.upper()
    return "BF16"

BYTES = {"BF16": 2, "FP8": 1, "INT8": 1, "AWQ": 0.5, "GPTQ": 0.5, "INT4": 0.5}

print(f"{'model':46} {'params':>7} {'prec':>6} {'weights GiB':>12}")
for m in models:
    p = params_b(m.id)
    if p is None or "VL" in m.id or "Embedding" in m.id:
        continue
    prec = precision(m.id)
    print(f"{m.id:46} {p:>6.1f}B {prec:>6} {p * BYTES[prec]:>12.1f}")

**Weights are the floor, not the total.** You also need room for the KV cache —
the per-token memory that holds attention state for every request in flight —
plus the CUDA context and activations. Budget the weights at roughly 60% of the
card and you will not be surprised.

FP8 halves the weights against BF16 and, on an H100 or newer, runs in native
hardware rather than being unpacked on the fly. That is why the default below is
an FP8 checkpoint.

In [ ]:
MODEL = "Qwen/Qwen3.8-27B-FP8"     # ~31GB of weights. Needs ~40GB free.
# Small alternative if you are on a 24GB card or just want this to finish fast:
# MODEL = "Qwen/Qwen3-0.6B"

MAX_MODEL_LEN = 32768              # context window we choose to serve
PORT = 8000

info = api.model_info(MODEL)
print("model    :", info.id)
print("license  :", (info.card_data or {}).get("license", "see model card"))
print("updated  :", info.last_modified)
print("downloads:", f"{info.downloads:,}")

---

## 2 · Install vLLM and serve the model

A serving engine does three things you would otherwise build yourself:

- packs many requests' KV caches into one pool of VRAM (**PagedAttention**)
- keeps the GPU busy as requests arrive and finish (**continuous batching**)
- speaks an API your code already understands (**the OpenAI protocol**)

The third is why swapping a hosted API for your own box is a base-URL change.

In [ ]:
%pip install -q vllm

In [ ]:
import vllm, torch
print("vllm  ", vllm.__version__)
print("torch ", torch.__version__)
print("cuda  ", torch.version.cuda)

### Baseline: what the GPU looks like before we load anything

Measure first. If something else is already holding VRAM — another notebook, a
container, a stopped-but-not-freed process — you want to know now, not from an
out-of-memory error five minutes into a weight download.

In [ ]:
import subprocess

def gpu_memory():
    """Used and total VRAM in MiB, per device."""
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True).stdout.strip()
    rows = []
    for line in out.splitlines():
        idx, name, used, total = [x.strip() for x in line.split(",")]
        rows.append((int(idx), name, int(used), int(total)))
    return rows

BEFORE = gpu_memory()
for idx, name, used, total in BEFORE:
    print(f"GPU {idx}  {name:28} {used:>6} / {total:>6} MiB used")

### Start the server

Every flag below has a failure mode, and three of them fail **silently** — no
error, just worse answers or tools that never fire.

In [ ]:
import os, subprocess, sys

LOG = "/tmp/vllm.log"

cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL,
    # How much of the card vLLM may claim. Too low and requests queue for KV
    # cache; too high and you OOM during startup.
    "--gpu-memory-utilization", "0.90",
    # SILENT FAILURE: left unset, vLLM derives the model's native context
    # (262k here) and the KV cache then fits roughly one request.
    "--max-model-len", str(MAX_MODEL_LEN),
    # How many requests may run concurrently. Too high, constant preemption;
    # too low, an idle GPU.
    "--max-num-seqs", "32",
    "--port", str(PORT),
]

# Qwen3 chat models need their tool and reasoning parsers named explicitly.
# SILENT FAILURE, both: the wrong tool parser returns tool calls as plain text
# so they never fire, and without the reasoning parser the thinking tokens leak
# into content and break any JSON you try to parse.
if "Qwen3" in MODEL and "0.6B" not in MODEL:
    cmd += ["--tool-call-parser", "qwen3_coder", "--reasoning-parser", "qwen3"]

print(" \\\n  ".join(cmd))

# vLLM compiles CUDA helpers at startup and shells out to `ninja` and a C
# compiler to do it. Those live in this environment's bin directory, which is
# NOT on PATH unless the venv was activated -- and a Jupyter kernel usually was
# not. Without this the server dies with FileNotFoundError: 'ninja', several
# hundred log lines below the line that actually mattered.
env = dict(os.environ)
env["PATH"] = os.path.dirname(sys.executable) + os.pathsep + env.get("PATH", "")

logfile = open(LOG, "w")
# start_new_session puts the server in its own process group, so the
# shutdown cell can signal it AND its engine-core child together.
server = subprocess.Popen(cmd, stdout=logfile, stderr=subprocess.STDOUT,
                          env=env, start_new_session=True)
print(f"\nstarted pid {server.pid}, logging to {LOG}")

### Wait for it to be ready

A cold start downloads the weights — about 31GB for the default model, so five
minutes or more on a good link. A warm start from the local cache is closer to
twenty seconds.

This polls `/health` rather than sleeping a fixed amount, and prints the last
log line so you can see what it is doing.

In [ ]:
import time, urllib.request, urllib.error

def wait_for_server(port=PORT, timeout=1800):
    started = time.time()
    while time.time() - started < timeout:
        if server.poll() is not None:
            print(open(LOG).read()[-2000:])
            raise RuntimeError(f"server exited with code {server.returncode}")
        try:
            urllib.request.urlopen(f"http://localhost:{port}/health", timeout=2)
            return time.time() - started
        except Exception:
            tail = [l for l in open(LOG).read().splitlines() if l.strip()]
            if tail:
                print(f"\r{int(time.time()-started):>4}s  {tail[-1][:96]:<96}", end="")
            time.sleep(5)
    raise TimeoutError("server did not become ready")

startup = wait_for_server()
print(f"\n\nready in {startup:.0f}s")

---

## 3 · Inference, memory and latency

### What it is serving

In [ ]:
import json, urllib.request

with urllib.request.urlopen(f"http://localhost:{PORT}/v1/models", timeout=10) as r:
    served = json.load(r)["data"][0]

print("served as    :", served["id"])
print("max_model_len:", served["max_model_len"])

### First inference

vLLM speaks the OpenAI protocol, so the official client works unchanged. The
`base_url` is the only thing that differs from talking to a hosted API — and the
API key is required by the client but ignored by the server.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=f"http://localhost:{PORT}/v1", api_key="not-used")

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "In two sentences: what is a KV cache?"}],
    temperature=0.2,
    max_tokens=200,
)
print(response.choices[0].message.content)
print("\ntokens:", response.usage.prompt_tokens, "in /",
      response.usage.completion_tokens, "out")

### How much GPU memory it is consuming

In [ ]:
AFTER = gpu_memory()

print(f"{'GPU':>3}  {'before':>10} {'after':>10} {'consumed':>10} {'total':>10}")
for (i, name, ub, tot), (_, _, ua, _) in zip(BEFORE, AFTER):
    print(f"{i:>3}  {ub:>7} MiB {ua:>7} MiB {ua-ub:>7} MiB {tot:>7} MiB")

used = AFTER[0][2] - BEFORE[0][2]
print(f"\nvLLM is holding {used/1024:.1f} GiB "
      f"({used/AFTER[0][3]:.0%} of the card).")

That single number hides three very different things. vLLM prints the split into
the log at startup — weights, activation peak, and whatever is left over becomes
the KV cache.

In [ ]:
import re

log = open(LOG).read()
for line in log.splitlines():
    if re.search(r"(model weights|KV cache|memory profiling|graph capturing|"
                 r"maximum concurrency)", line, re.I):
        print(line.split("] ")[-1].strip()[:150])

**The KV cache is the part that scales with your users.** Weights are a fixed
cost paid once. The KV cache is per-token, per-request, and it is what decides
how many people can talk to this box at the same time.

In [ ]:
# Bytes of KV cache per token = 2 (one K, one V) x layers x kv_heads
#                               x head_dim x bytes_per_element
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained(MODEL, trust_remote_code=True)
layers   = getattr(cfg, "num_hidden_layers", None)
kv_heads = getattr(cfg, "num_key_value_heads", None) or getattr(cfg, "num_attention_heads")
heads    = getattr(cfg, "num_attention_heads")
head_dim = getattr(cfg, "head_dim", None) or cfg.hidden_size // heads

kv_per_token = 2 * layers * kv_heads * head_dim * 2      # BF16 cache = 2 bytes
per_request  = kv_per_token * MAX_MODEL_LEN

print(f"layers                 : {layers}")
print(f"attention / kv heads   : {heads} / {kv_heads}   (GQA divides by "
      f"{heads // kv_heads})")
print(f"head_dim               : {head_dim}")
print(f"KV per token           : {kv_per_token/1024:.1f} KiB")
print(f"one full {MAX_MODEL_LEN}-token request : {per_request/1024**3:.2f} GiB")

> **Note.** If the model uses sliding-window or hybrid attention, only the
> full-attention layers keep a cache and the real figure is lower than the
> formula above. vLLM's own log line is the authority; the arithmetic is here so
> the number is not magic.

### Latency

One request at a time, which is the *worst* case for a GPU — the card is mostly
idle waiting for the next token.

In [ ]:
import time, statistics

PROMPT = "Explain what continuous batching does, in three sentences."
latencies, out_tokens = [], []

for i in range(5):
    t0 = time.perf_counter()
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": PROMPT}],
        temperature=0.0, max_tokens=150,
    )
    latencies.append(time.perf_counter() - t0)
    out_tokens.append(r.usage.completion_tokens)
    print(f"run {i+1}: {latencies[-1]:.2f}s  {out_tokens[-1]} tokens out")

median = statistics.median(latencies)
print(f"\nmedian latency : {median:.2f}s")
print(f"throughput     : {statistics.median(out_tokens)/median:.0f} tokens/sec "
      f"at concurrency 1")

### The same work, concurrently

This is the whole argument for a serving engine. The GPU is not busier because
you asked politely — it is busier because continuous batching keeps it fed.

In [ ]:
import concurrent.futures as cf

def one_call():
    t0 = time.perf_counter()
    r = client.chat.completions.create(
        model=MODEL, messages=[{"role": "user", "content": PROMPT}],
        temperature=0.0, max_tokens=150)
    return time.perf_counter() - t0, r.usage.completion_tokens

print(f"{'concurrent':>10} {'tok/s':>9} {'median latency':>16}")
for n in (1, 4, 8):
    t0 = time.perf_counter()
    with cf.ThreadPoolExecutor(max_workers=n) as pool:
        results = list(pool.map(lambda _: one_call(), range(n)))
    wall = time.perf_counter() - t0
    total_out = sum(t for _, t in results)
    med = statistics.median(l for l, _ in results)
    print(f"{n:>10} {total_out/wall:>9.0f} {med:>15.2f}s")

Latency per request goes **up** and total throughput goes **up much more**. That
trade is the entire economics of owning the card: an idle GPU is the most
expensive inference there is.

---

## 4 · Traceability

An answer you cannot explain is an answer you cannot defend. The moment someone
asks *"why did it say that?"* you need the prompt actually sent, the parameters,
the token counts and the latency — for **one specific request**, not an
aggregate.

Logs cannot do this. A log line per component gives you fragments with nothing
tying them together. A trace is the thread.

**Phoenix** is an open-source trace viewer that speaks OpenTelemetry. It runs
locally, stores traces locally, and nothing leaves the machine.

In [ ]:
# The EXPORTER only, not the Phoenix server.
#
# `pip install arize-phoenix` pulls the whole server, and next to vLLM it
# resolves to a version whose own dependency (pydantic-ai) then fails to
# import -- ImportError: cannot import name 'MCPServerStreamableHTTP'.
# Measured here, on this box.
#
# You do not want the trace backend inside your application process anyway.
%pip install -q arize-phoenix-otel openinference-instrumentation-openai

### Launch the viewer

This starts Phoenix in this process and returns its URL. On a remote box, forward
the port (`ssh -L 6006:localhost:6006 user@host`) or open it on the box's own
address.

Phoenix runs as a **service**, not a library. Start it once, in a terminal:

```bash
docker run -d --name phoenix --restart unless-stopped -p 6006:6006 \
  -e PHOENIX_WORKING_DIR=/data -v phoenix-data:/data \
  arizephoenix/phoenix:latest
```

The volume matters: without it a restart discards the traces you were about to
look at. Then the next cell waits for it.

In [ ]:
import time, urllib.request

PHOENIX_HOST = "localhost"        # the box you are running this on
PHOENIX_PORT = 6006

for attempt in range(30):
    try:
        urllib.request.urlopen(f"http://{PHOENIX_HOST}:{PHOENIX_PORT}/", timeout=3)
        print(f"Phoenix is up: http://{PHOENIX_HOST}:{PHOENIX_PORT}/")
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError(
        "Phoenix is not answering on that port. Start the container above, "
        "then re-run this cell."
    )

### Point OpenTelemetry at it, and instrument the client

Two calls. `register` wires up the exporter; `OpenAIInstrumentor` patches the
OpenAI client so every `chat.completions.create` becomes a span, with no change
to the calling code.

`batch=False` sends spans immediately instead of buffering — right for a demo
where you refresh the UI, wrong for production.

In [ ]:
from openinference.instrumentation.openai import OpenAIInstrumentor
from phoenix.otel import register

PROJECT = "serving-notebook"

provider = register(
    endpoint=f"http://{PHOENIX_HOST}:{PHOENIX_PORT}/v1/traces",
    project_name=PROJECT,
    # batch=False sends each span as it ends, so it shows up in the UI
    # immediately. Right for a demo you are watching, wrong for production.
    batch=False,
    set_global_tracer_provider=True,
)
OpenAIInstrumentor().instrument(tracer_provider=provider)

from opentelemetry import trace
tracer = trace.get_tracer(__name__)
print("tracing on, project:", PROJECT)

### A traced call

Wrapping the call in a span of our own gives the trace a root to hang from — the
model call nests inside it. Anything else you add later (retrieval, tools,
guards) becomes a sibling under the same root.

In [ ]:
def trace_id_hex():
    ctx = trace.get_current_span().get_span_context()
    return format(ctx.trace_id, "032x") if ctx.trace_id else None

with tracer.start_as_current_span("chat") as span:
    span.set_attribute("user.question", "What is PagedAttention?")
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "What is PagedAttention, in two sentences?"}],
        temperature=0.2, max_tokens=200,
    )
    tid = trace_id_hex()

print(r.choices[0].message.content)
print("\ntrace_id:", tid)

### Read that trace back, without opening the UI

In [ ]:
import json, time, urllib.request

time.sleep(3)          # spans export as they END, and the root ends LAST

def fetch_spans(project=PROJECT, limit=200):
    """Straight off the REST API. No client library, no version to match."""
    url = f"http://{PHOENIX_HOST}:{PHOENIX_PORT}/v1/projects/{project}/spans?limit={limit}"
    with urllib.request.urlopen(url, timeout=30) as r:
        return json.load(r)["data"]

spans = fetch_spans()
mine = [s for s in spans if s["context"]["trace_id"] == tid]
print(f"{len(spans)} spans in the project, {len(mine)} in this trace\n")
print(f"{'span':24} {'kind':<12} ms")
for s in sorted(mine, key=lambda s: s["start_time"]):
    import datetime
    f = "%Y-%m-%dT%H:%M:%S.%f%z"
    ms = (datetime.datetime.strptime(s["end_time"], f)
          - datetime.datetime.strptime(s["start_time"], f)).total_seconds() * 1000
    print(f"{s['name']:24} {s.get('span_kind',''):<12} {ms:8.1f}")

### How to view a specific run in the UI

1. Open the Phoenix URL printed above.
2. Choose the **serving-notebook** project.
3. Paste the `trace_id` into the search box — it is the 32-character hex string
   printed with the answer.
4. Click the trace. You get a tree: your `chat` span with the `ChatCompletion`
   nested inside it.

Open the model-call span and look at the attributes. The **prompt actually
sent** is there, and it is always bigger than people expect once the chat
template and any history are counted. So are the token counts and the latency —
which is how a "why is this slow" argument turns into a measurement.

> **Spans export as they end, and the root span ends last.** Query too fast and
> you get children with no root, which renders as an empty tree and looks
> exactly like broken tracing. Wait a second before refreshing.

### Why we are not using the Hub

Guardrails AI ships **zero** validators. All ~65 live in a Hub fetched from
`hub.api.guardrailsai.com`, one `guardrails hub install` at a time. Run it on a
box where that host does not resolve and you get:

```
Failed to resolve 'hub.api.guardrailsai.com'
```

which is exactly the network a data-residency argument implies. Measured on this
box — the install simply fails.

So below we register **our own validators**, using the framework properly —
`@register_validator`, a `Validator` subclass, `Guard().use(...)`,
`on_fail="exception"` — backed by two engines installed from PyPI:

| Validator | Engine | What it catches |
|---|---|---|
| `PresidioPII` | Presidio + spaCy NER | names, places, emails, cards — *the same engine the Hub's `DetectPII` wraps* |
| `ToxicText` | `unitary/toxic-bert` | toxic / NSFW output |

Same framework, same semantics, no Hub, no token, no egress.

In [ ]:
%pip install -q guardrails-ai presidio-analyzer transformers
# Presidio needs a spaCy model and does not fetch one itself.
%pip install -q https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl

In [ ]:
import os

# Set BEFORE importing guardrails: both are read at import time.
os.environ["GUARDRAILS_DISABLE_TELEMETRY"] = "true"
os.environ["GUARDRAILS_ENABLE_METRICS"] = "false"

import guardrails  # noqa: F401
from importlib.metadata import version

# The package exposes no __version__ attribute -- ask the installed metadata.
print("guardrails-ai", version("guardrails-ai"))
print("presidio-analyzer", version("presidio-analyzer"))

### Install validators from the Hub

These are separate downloads. `DetectPII` pulls in Presidio and a spaCy model;
`NSFWText` pulls a small classifier from Hugging Face. Expect a few minutes and
a few hundred MB.

If your Hub account needs configuring, run `guardrails configure` in a terminal
first and paste the token from https://hub.guardrailsai.com.

In [ ]:
!guardrails hub install hub://guardrails/detect_pii --quiet
!guardrails hub install hub://guardrails/nsfw_text --quiet

In [ ]:
from guardrails import Guard
from guardrails.classes.validation.validation_result import (
    FailResult, PassResult, ValidationResult,
)
from guardrails.validator_base import Validator, register_validator


@register_validator(name="local/presidio-pii", data_type="string")
class PresidioPII(Validator):
    """Personal data, via NER -- so it sees what has no pattern."""

    ENTITIES = ("PERSON", "EMAIL_ADDRESS", "PHONE_NUMBER", "CREDIT_CARD",
                "US_SSN", "LOCATION")

    def __init__(self, threshold=0.5, **kwargs):
        super().__init__(**kwargs)
        self.threshold = threshold
        from presidio_analyzer import AnalyzerEngine
        from presidio_analyzer.nlp_engine import NlpEngineProvider
        provider = NlpEngineProvider(nlp_configuration={
            "nlp_engine_name": "spacy",
            "models": [{"lang_code": "en", "model_name": "en_core_web_sm"}],
        })
        self.analyzer = AnalyzerEngine(nlp_engine=provider.create_engine())

    def _validate(self, value: str, metadata: dict) -> ValidationResult:
        hits = [r for r in self.analyzer.analyze(
                    text=value, entities=list(self.ENTITIES), language="en")
                if r.score >= self.threshold]
        if not hits:
            return PassResult()
        worst = max(hits, key=lambda r: r.score)
        # Report the CATEGORY, never the value -- a message that quotes the
        # personal data it refused has just leaked it into your logs.
        return FailResult(error_message=(
            f"personal data detected: {worst.entity_type} "
            f"(confidence {worst.score:.2f})"))


@register_validator(name="local/toxic-text", data_type="string")
class ToxicText(Validator):
    """Toxic or NSFW text, via a small classifier."""

    def __init__(self, threshold=0.8, **kwargs):
        super().__init__(**kwargs)
        self.threshold = threshold
        from transformers import pipeline
        self.clf = pipeline("text-classification",
                            model="unitary/toxic-bert", top_k=None)

    def _validate(self, value: str, metadata: dict) -> ValidationResult:
        scores = {d["label"]: d["score"] for d in self.clf(value)[0]}
        label, score = max(scores.items(), key=lambda kv: kv[1])
        if score < self.threshold:
            return PassResult()
        return FailResult(error_message=f"{label} content ({score:.2f})")


# on_fail belongs on the VALIDATOR, not on .use() -- put it on .use() and it is
# silently ignored, the Guard returns a failed outcome instead of raising, and
# your guard looks like it never fires.
pii_guard  = Guard().use(PresidioPII(threshold=0.5, on_fail="exception"))
nsfw_guard = Guard().use(ToxicText(threshold=0.8, on_fail="exception"))
print("validators registered and loaded")

### PII on the way in

In [ ]:
CASES = [
    "How do I reset my password?",
    "My email is jane.doe@corplabs.com, please help",
    "Call me on +44 7700 900123 when the ticket is fixed",
    "I am Jane Doe from the Manchester office and I need VPN access",
]

for text in CASES:
    try:
        pii_guard.validate(text)
        print(f"PASS    {text}")
    except Exception as error:
        reason = str(error).split("\n")[0][:80]
        print(f"REFUSED {text}\n        -> {reason}")

Note which one is interesting. An email is a **pattern** and a regex would catch
it too. *"I am Jane Doe from the Manchester office"* has no pattern in it at all
— catching that is named-entity recognition, and it is the thing a regex
structurally cannot do.

That is the honest case for the framework: not that it wraps your checks more
neatly, but that Presidio finds categories your regex was never going to reach.

### NSFW on the way out

Same shape, different validator. This one is a small classifier rather than a
rule, so it costs milliseconds instead of microseconds.

In [ ]:
OUTPUTS = [
    "Go to https://passwordreset.corplabs.com and approve the MFA prompt.",
    "I hate you and everyone in this miserable building.",
]

for text in OUTPUTS:
    try:
        nsfw_guard.validate(text)
        print(f"PASS    {text[:70]}")
    except Exception as error:
        print(f"REFUSED {text[:70]}\n        -> {str(error).split(chr(10))[0][:80]}")

### Put the guards around the traced call

Now the three pieces come together. Input guard, model, output guard — each in
its own span, so a refusal is attached to the request that caused it rather than
sitting in a log file you have to correlate by timestamp.

In [ ]:
class Refused(Exception):
    """A guard said no. Carries which guard, and why."""


def guarded_chat(question: str, max_tokens: int = 200) -> dict:
    with tracer.start_as_current_span("chat") as root:
        root.set_attribute("user.question", question)
        tid = trace_id_hex()

        # --- input guard ---------------------------------------------------
        with tracer.start_as_current_span("input_guard") as span:
            span.set_attribute("guardrail.name", "detect_pii")
            try:
                pii_guard.validate(question)
                span.set_attribute("guardrail.passed", True)
            except Exception as error:
                reason = str(error).split("\n")[0][:160]
                span.set_attribute("guardrail.passed", False)
                span.set_attribute("guardrail.reason", reason)
                return {"refused_by": "input_guard", "reason": reason, "trace_id": tid}

        # --- the model -----------------------------------------------------
        answer = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": question}],
            temperature=0.2, max_tokens=max_tokens,
        ).choices[0].message.content

        # --- output guard --------------------------------------------------
        with tracer.start_as_current_span("output_guard") as span:
            span.set_attribute("guardrail.name", "nsfw_text")
            try:
                nsfw_guard.validate(answer)
                span.set_attribute("guardrail.passed", True)
            except Exception as error:
                reason = str(error).split("\n")[0][:160]
                span.set_attribute("guardrail.passed", False)
                span.set_attribute("guardrail.reason", reason)
                return {"refused_by": "output_guard", "reason": reason, "trace_id": tid}

        return {"answer": answer, "trace_id": tid}

In [ ]:
for q in ["What is continuous batching?",
          "My email is jane.doe@corplabs.com — what is my ticket status?"]:
    result = guarded_chat(q)
    if "answer" in result:
        print(f"ANSWERED  {q}\n          {result['answer'][:110]}...")
    else:
        print(f"REFUSED   {q}\n          by {result['refused_by']}: {result['reason'][:70]}")
    print(f"          trace {result['trace_id']}\n")

### Measure what the guards cost

Never put a guard in front of every request without knowing its price.

In [ ]:
import statistics, time

def time_guard(guard, text, n=20):
    samples = []
    for _ in range(n):
        t0 = time.perf_counter()
        try:
            guard.validate(text)
        except Exception:
            pass
        samples.append((time.perf_counter() - t0) * 1000)
    return statistics.median(samples)

print(f"{'guard':14} {'median ms':>10}")
print(f"{'detect_pii':14} {time_guard(pii_guard, CASES[0]):>10.1f}")
print(f"{'nsfw_text':14} {time_guard(nsfw_guard, OUTPUTS[0]):>10.1f}")

Compare that with the model call, which is measured in **seconds**. Guards are
cheap next to inference — but they run on every request, including the ones they
never block, so the number is worth knowing rather than assuming.

> **What these guards do not catch.** A validator is not a security boundary.
> Published defences against prompt injection have been broken more than 90% of
> the time once attackers adapted to them, and guard models vary enormously in
> recall. Treat these as a filter that removes the obvious, not a wall.

Open Phoenix again and look at one of the refused requests. The
`input_guard` span carries `guardrail.passed = false` and the reason, nested
under the same root as the request that caused it. That is what makes a
guardrail debuggable instead of merely present.

---

## Shut it down

The server holds the GPU until you stop it. A forgotten process is the most
common way to waste a rented card.

In [ ]:
import signal, time

# vLLM runs an API server process plus an engine-core child. Terminating the
# parent alone leaves the child holding the VRAM, and a naive check straight
# afterwards still reports the card as full -- which looks like a leak and is
# really just impatience. Signal the whole process group, then WAIT for the
# memory to actually come back.
try:
    os.killpg(os.getpgid(server.pid), signal.SIGTERM)
except (ProcessLookupError, PermissionError):
    server.terminate()

try:
    server.wait(timeout=60)
except subprocess.TimeoutExpired:
    print("did not exit on SIGTERM, sending SIGKILL")
    os.killpg(os.getpgid(server.pid), signal.SIGKILL)
    server.wait(timeout=30)

# SIGTERM makes the exit code non-zero. That is the signal, not a fault.
print(f"vLLM stopped (exit {server.returncode}, negative means killed by signal)")

for _ in range(30):
    used = gpu_memory()[0][2]
    if used < 2000:
        break
    time.sleep(2)

for idx, name, used, total in gpu_memory():
    print(f"GPU {idx}  {used:>6} / {total:>6} MiB used")

---

## What you built

| Step | What it proved |
|---|---|
| Listed the registry | You choose the checkpoint, and you can pin it |
| Served it with vLLM | An OpenAI-compatible endpoint is a base-URL change |
| Measured memory | Weights are fixed; the KV cache is what scales with users |
| Measured latency | Concurrency raises latency a little and throughput a lot |
| Added tracing | One request, explainable, with the real prompt attached |
| Added guardrails | Checks around the model, in spans, with measured cost |

Nothing above required a hosted API, and no prompt left the machine.

**If you are renting the GPU, stop the instance now** — not just the server.